In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/

/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능


In [ ]:
!pip install transformers
!pip install torchinfo
!pip install pytorch-lightning
!pip install torchmetrics
!pip install sentencepiece
!pip install -i https://test.pypi.org/simple/ bitsandbytes
!pip install peft
!pip install accelerate
!pip install -q -U datasets scipy ipywidgets

In [ ]:
import numpy as np
import pandas as pd
import os, sys, re, math, random, time, json, pickle, tqdm, gc
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import AdamW, get_linear_schedule_with_warmup
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from tqdm import tqdm

class args:
    # amp = True
    gpu = '0'

    label_size = 2
    epochs=10
    batch_size=8
    weight_decay=1e-6
    max_len = 60
    fold = 5

    start_lr = 2e-5

    num_workers=0
    seed=2022

os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu
device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
print(device)
##----------------
def set_seeds(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seeds(seed=args.seed)

path = '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/'

df_train = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/df_train.csv', encoding = 'utf-8')
df_dev = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/df_dev.csv', encoding = 'utf-8')
df_test = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/df_test.csv', encoding = 'utf-8')

df_train.dropna(inplace = True)
df_dev.dropna(inplace = True)
df_test.dropna(inplace = True)

df_train.reset_index(drop = True, inplace = True)
df_dev.reset_index(drop = True, inplace = True)

cuda


In [ ]:
from transformers import BitsAndBytesConfig

base_model_id = "beomi/KoAlpaca-Polyglot-12.8B"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForSequenceClassification.from_pretrained(base_model_id, use_cache=False, quantization_config=bnb_config, num_labels = args.label_size)

config.json:   0%|          | 0.00/682 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/52.5k [00:00<?, ?B/s]

model-00001-of-00028.safetensors:   0%|          | 0.00/945M [00:00<?, ?B/s]

model-00002-of-00028.safetensors:   0%|          | 0.00/843M [00:00<?, ?B/s]

model-00003-of-00028.safetensors:   0%|          | 0.00/843M [00:00<?, ?B/s]

...

model-00028-of-00028.safetensors:   0%|          | 0.00/517M [00:00<?, ?B/s]


===================================BUG REPORT===================================
Welcome to bitsandbytes. For bug reports, please run

python -m bitsandbytes

 and submit this information together with your error trace to: https://github.com/TimDettmers/bitsandbytes/issues
CUDA_SETUP: WARNING! libcudart.so not found in any environmental path. Searching in backup paths...
CUDA SETUP: CUDA runtime path found: /usr/local/cuda/lib64/libcudart.so.11.0
CUDA SETUP: Highest compute capability among GPUs detected: 8.0
CUDA SETUP: Detected CUDA version 118
CUDA SETUP: Loading binary /usr/local/lib/python3.10/dist-packages/bitsandbytes/libbitsandbytes_cuda118.so...


/usr/local/lib/python3.10/dist-packages/bitsandbytes/cuda_setup/main.py:147: UserWarning: /usr/lib64-nvidia did not contain ['libcudart.so', 'libcudart.so.11.0', 'libcudart.so.12.0'] as expected! Searching further paths...
  warn(msg)
/usr/local/lib/python3.10/dist-packages/bitsandbytes/cuda_setup/main.py:147: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('/sys/fs/cgroup/memory.events /var/colab/cgroup/jupyter-children/memory.events')}
  warn(msg)
/usr/local/lib/python3.10/dist-packages/bitsandbytes/cuda_setup/main.py:147: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('8013'), PosixPath('http'), PosixPath('//172.28.0.1')}
  warn(msg)
/usr/local/lib/python3.10/dist-packages/bitsandbytes/cuda_setup/main.py:147: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('//colab.research.google.com/tun/m/cc483011

Loading checkpoint shards:   0%|          | 0/28 [00:00<?, ?it/s]

Some weights of GPTNeoXForSequenceClassification were not initialized from the model checkpoint at beomi/KoAlpaca-Polyglot-12.8B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
class CustomDataset(Dataset):
    def __init__(self, dataset, tokenizer, max_len):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        text = self.dataset.iloc[idx]['input']
        label = self.dataset.iloc[idx]['output']
        prompt = f"### 질문: 문장에 혐오 표현이 사용되었는지 확인하세요. 사용되었다면 1, 사용되지 않았다면 0을 출력하세요. '{text}'\n\n### 답변: {label}"

        inputs = self.tokenizer.encode_plus(prompt,
                                            return_tensors='pt',
                                            truncation=True,
                                            max_length=self.max_len,
                                            padding='max_length',
                                            add_special_tokens=True)

        input_ids = inputs['input_ids'][0]
        attention_mask = inputs['attention_mask'][0]

        return input_ids, attention_mask, label

    def __len__(self):
        return len(self.dataset)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

tokenizer.eos_token = tokenizer.eos_token
tokenizer.pad_token = tokenizer.pad_token if tokenizer.pad_token is not None else tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.65M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

In [ ]:
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, inference_mode=False, r=8, lora_alpha=16, lora_dropout=0.05, bias="none"
)

model = prepare_model_for_kbit_training(base_model)
peft_model = get_peft_model(model, peft_config)

dir = '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold0_KoAlpaca_Polyglot_12.8B_kfold.pt'
torch.save(peft_model.state_dict(), dir)

In [ ]:
import transformers
from datetime import datetime
import torch.optim as optim
import torch.nn.functional as F

peft_model.config.pad_token_id = tokenizer.pad_token_id

kf = KFold(n_splits=5, shuffle=True, random_state=42)

df = pd.concat([df_train, df_dev])

# KFold 교차 검증
fold = 0
for fold, (train_index, valid_index) in enumerate(kf.split(df)):
    print(f"Fold {fold + 1}/{args.fold}")

    peft_model.load_state_dict(torch.load(dir))

    df_train = df.iloc[train_index].reset_index(drop=True)
    df_valid = df.iloc[valid_index].reset_index(drop=True)

    train_dataset = CustomDataset(df_train, tokenizer, args.max_len)
    train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)
    valid_dataset = CustomDataset(df_valid, tokenizer, args.max_len)
    valid_loader = DataLoader(valid_dataset, batch_size=args.batch_size, shuffle=False)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = AdamW(peft_model.parameters(), lr=args.start_lr)
    scheduler = get_cosine_schedule_with_warmup(optimizer,
                                                num_warmup_steps = int(len(train_loader)*args.epochs * 0.1),
                                                num_training_steps = len(train_loader)*args.epochs)

    stop_val_f1 = 0
    stop_count = 0

    for epoch in range(args.epochs):
        if stop_count == 3:
            break
        peft_model.train()
        train_loss = 0
        target_lst = []
        pred_lst = []
        pbar = tqdm(train_loader)
        for ids, mask, target in pbar:
            ids = ids.to(device)
            mask = mask.to(device)
            target = target.to(device)

            optimizer.zero_grad()
            outputs = peft_model(ids, mask)
            loss = loss_fn(outputs.logits, target).to(device)
            train_loss += loss.item()

            loss.backward()
            optimizer.step()
            scheduler.step()

            target_lst.extend(target.detach().cpu().numpy())
            pred_lst.extend(outputs.logits.argmax(dim = 1).tolist())
            pbar.set_description('\033[1m[C_loss : {:>.5}]\033[0m'.format(round(train_loss / len(pbar), 4)))

        train_loss = train_loss / len(train_loader)
        train_score = f1_score(y_true=target_lst, y_pred=pred_lst, average='micro')
        train_acc = accuracy_score(y_true=target_lst, y_pred=pred_lst)

        torch.cuda.empty_cache()
        gc.collect()

        peft_model.eval()
        valid_loss = 0
        target_lst = []
        pred_lst = []
        pbar = tqdm(valid_loader)
        for ids, mask, target in pbar:
            ids = ids.to(device)
            mask = mask.to(device)
            target = target.to(device)

            outputs = peft_model(ids, mask)
            loss = loss_fn(outputs.logits, target).to(device)
            valid_loss += loss.item()

            target_lst.extend(target.detach().cpu().numpy())
            pred_lst.extend(outputs.logits.argmax(dim = 1).tolist())
            pbar.set_description('\033[1m[C_loss : {:>.5}]\033[0m'.format(round(valid_loss / len(pbar), 4)))
        valid_loss = valid_loss / len(valid_loader)
        valid_score = f1_score(y_true=target_lst, y_pred=pred_lst, average='micro')
        valid_acc = accuracy_score(y_true=target_lst, y_pred=pred_lst)

        torch.cuda.empty_cache()
        gc.collect()

        print(f'Fold {fold + 1}, Epoch : {epoch + 1},    t_loss : {round(train_loss, 4)},   t_f1 : {round(train_score, 4)},   t_acc : {round(train_acc, 4)}\n')
        print(f'                     v_loss : {round(valid_loss, 4)},   v_f1 : {round(valid_score, 4)},   v_acc : {round(valid_acc, 4)}\n')

        if valid_score > stop_val_f1:
            default_weight_path = path + f'fold{fold + 1}_KoAlpaca_Polyglot_12.8B_kfold.pt'
            torch.save(peft_model.state_dict(), default_weight_path)
            stop_val_f1 = valid_score
            stop_count = 0
        else:
            stop_count += 1

        torch.cuda.empty_cache()
        gc.collect()

Fold 1/5


/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
  0%|          | 0/1866 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
[C_loss : 0.3146]: 100%|██████████| 467/467 [01:25<00:00,  5.47it/s]


Fold 1, Epoch : 1,    t_loss : 0.7621,   t_f1 : 0.7811,   t_acc : 0.7811

                     v_loss : 0.3146,   v_f1 : 0.8955,   v_acc : 0.8955



  0%|          | 0/1866 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
[C_loss : 0.1739]: 100%|██████████| 467/467 [01:25<00:00,  5.46it/s]


Fold 1, Epoch : 2,    t_loss : 0.174,   t_f1 : 0.9374,   t_acc : 0.9374

                     v_loss : 0.1739,   v_f1 : 0.9434,   v_acc : 0.9434



...

Fold 5, Epoch : 8,    t_loss : 0.0001,   t_f1 : 1.0,   t_acc : 1.0

                     v_loss : 0.1718,   v_f1 : 0.9617,   v_acc : 0.9617



  0%|          | 0/1866 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
[C_loss : 0.1747]: 100%|██████████| 467/467 [01:25<00:00,  5.47it/s]


Fold 5, Epoch : 9,    t_loss : 0.0001,   t_f1 : 1.0,   t_acc : 1.0

                     v_loss : 0.1747,   v_f1 : 0.9619,   v_acc : 0.9619



  0%|          | 0/1866 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
[C_loss : 0.1752]: 100%|██████████| 467/467 [01:25<00:00,  5.46it/s]


Fold 5, Epoch : 10,    t_loss : 0.0001,   t_f1 : 1.0,   t_acc : 1.0

                     v_loss : 0.1752,   v_f1 : 0.9619,   v_acc : 0.9619



In [ ]:
from transformers import BitsAndBytesConfig

base_model_id = "beomi/KoAlpaca-Polyglot-12.8B"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForSequenceClassification.from_pretrained(base_model_id, use_cache=False, quantization_config=bnb_config, num_labels = args.label_size)


===================================BUG REPORT===================================
Welcome to bitsandbytes. For bug reports, please run

python -m bitsandbytes

 and submit this information together with your error trace to: https://github.com/TimDettmers/bitsandbytes/issues
CUDA_SETUP: WARNING! libcudart.so not found in any environmental path. Searching in backup paths...
CUDA SETUP: CUDA runtime path found: /usr/local/cuda/lib64/libcudart.so
CUDA SETUP: Highest compute capability among GPUs detected: 8.0
CUDA SETUP: Detected CUDA version 118
CUDA SETUP: Loading binary /usr/local/lib/python3.10/dist-packages/bitsandbytes/libbitsandbytes_cuda118.so...


/usr/local/lib/python3.10/dist-packages/bitsandbytes/cuda_setup/main.py:147: UserWarning: /usr/lib64-nvidia did not contain ['libcudart.so', 'libcudart.so.11.0', 'libcudart.so.12.0'] as expected! Searching further paths...
  warn(msg)
/usr/local/lib/python3.10/dist-packages/bitsandbytes/cuda_setup/main.py:147: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('/sys/fs/cgroup/memory.events /var/colab/cgroup/jupyter-children/memory.events')}
  warn(msg)
/usr/local/lib/python3.10/dist-packages/bitsandbytes/cuda_setup/main.py:147: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('http'), PosixPath('8013'), PosixPath('//172.28.0.1')}
  warn(msg)
/usr/local/lib/python3.10/dist-packages/bitsandbytes/cuda_setup/main.py:147: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('//colab.research.google.com/tun/m/cc483011

Loading checkpoint shards:   0%|          | 0/28 [00:00<?, ?it/s]

Some weights of GPTNeoXForSequenceClassification were not initialized from the model checkpoint at beomi/KoAlpaca-Polyglot-12.8B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
class TestDataset(Dataset):
    def __init__(self, dataset, tokenizer, max_len):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        text = self.dataset.iloc[idx]['input']
        prompt = f"### 질문: 문장에 혐오 표현이 사용되었는지 확인하세요. 사용되었다면 1, 사용되지 않았다면 0을 출력하세요. '{text}'\n\n### 답변: "

        inputs = self.tokenizer.encode_plus(prompt,
                                            return_tensors='pt',
                                            truncation=True,
                                            max_length=self.max_len,
                                            padding='max_length',
                                            add_special_tokens=True)

        input_ids = inputs['input_ids'][0]
        attention_mask = inputs['attention_mask'][0]

        return input_ids, attention_mask

    def __len__(self):
        return len(self.dataset)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

tokenizer.eos_token = tokenizer.eos_token
tokenizer.pad_token = tokenizer.pad_token if tokenizer.pad_token is not None else tokenizer.eos_token

test_dataset = TestDataset(df_test, tokenizer, args.max_len)
test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False)

In [ ]:
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, inference_mode=False, r=8, lora_alpha=16, lora_dropout=0.05, bias="none"
)

model = prepare_model_for_kbit_training(base_model)
peft_model = get_peft_model(model, peft_config)

In [ ]:
peft_model.config.pad_token_id = tokenizer.pad_token_id

model_paths = ['/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold1_KoAlpaca_Polyglot_12.8B_kfold.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold2_KoAlpaca_Polyglot_12.8B_kfold.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold3_KoAlpaca_Polyglot_12.8B_kfold.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold4_KoAlpaca_Polyglot_12.8B_kfold.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold5_KoAlpaca_Polyglot_12.8B_kfold.pt']

models = []
for model_path in model_paths:
    peft_model.load_state_dict(torch.load(model_path))
    peft_model.to(device)
    peft_model.eval()
    models.append(peft_model)

res = []
with torch.no_grad():
    for ids, mask in tqdm(test_loader):
        ids = ids.to(device)
        mask = mask.to(device)

        outputs = [peft_model(ids, mask).logits for model in models]
        avg_output = sum(outputs) / len(models)
        res.extend(avg_output.argmax(axis=1).tolist())

df_test['output'] = res

100%|██████████| 259/259 [03:48<00:00,  1.13it/s]


In [ ]:
df_test

,Unnamed: 0,id,input,output
0,0,nikluge-au-2022-test-000001,극좌는 이 비겁자층을 제대로 요리할 줄 안다...,1
1,1,nikluge-au-2022-test-000002,내가 진짜 올 해 안에 차 산다!,0
2,2,nikluge-au-2022-test-000003,"선거 때마다 불장난 하는 못된 버릇 대대손손 배워가지고 그러고 까불어대면, 너 나중...",1
3,3,nikluge-au-2022-test-000004,"난 99년도에 이미 세상은 망해서 선한자들은 이미 모두 하늘로 올라갔고, 남은 우리...",1
4,4,nikluge-au-2022-test-000005,피의자로 가는 싸가지 없는 쓰래기!,1
...,...,...,...,...
2067,2067,nikluge-au-2022-test-002068,근데 비계 올 사람만 찍어,0
2068,2068,nikluge-au-2022-test-002069,지들 입맛대로 막 가네 미친,1
2069,2069,nikluge-au-2022-test-002070,얘는 지가 이걸 쓰면서 왜 이런 생각을 못하지?,0
2070,2070,nikluge-au-2022-test-002071,알겟나요,0


In [ ]:
def jsonlload(fname):
    with open(fname, "r", encoding="utf-8") as f:
        lines = f.read().strip().split("\n")
        j_list = [json.loads(line) for line in lines]
    return j_list

test_df = jsonlload('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/NIKL_AU_2023_COMPETITION_v1.0/nikluge-au-2022-test.jsonl')
len(test_df)

2072

In [ ]:
for i, p in enumerate(df_test['output']):
    test_df[i]['output'] = p

In [ ]:
def jsonldump(j_list, fname):
    with open(fname, "w", encoding='utf-8') as f:
        for json_data in j_list:
            f.write(json.dumps(json_data, ensure_ascii=False)+'\n')
jsonldump(test_df, '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/test25_kfold_koalpaca_polyglot_12.8b.jsonl')